# BEAUTYPRICE INTELLIGENCE — Bộ KPI
Notebook riêng cho Nhiệm vụ 2, Giai đoạn 6. KPI được chốt sau EDA và không nằm chung trong notebook EDA.

In [1]:
from pathlib import Path
import runpy, sys, pandas as pd
candidates=[Path.cwd(),Path.cwd()/'BDr'/'kqptr',Path.cwd()/'kqptr']
folder=next(p for p in candidates if (p/'02_kpi.py').is_file())
sys.path.insert(0,str(folder.resolve()))
g=runpy.run_path(str(folder/'02_kpi.py'))

Created: C:\Users\Latitude 5500\OneDrive\Big Data\BDr\kqptr\02_kpi_dictionary.xlsx


## Từ điển KPI
Mỗi KPI có định nghĩa nghiệp vụ, công thức, đơn vị, điều kiện lọc, phạm vi thời gian, nguồn và phương pháp kiểm tra. Tỷ lệ lưu dạng 0..1.

In [2]:
kpi=pd.DataFrame(g['rows'],columns=g['headers'])
kpi

kpi_id,kpi_name,business_definition,formula,unit,filter_conditions,time_scope,current_value,status,data_source,validation_method,reason
KPI13,Tổng sản phẩm duy nhất,Sản phẩm duy nhất trong bảng chuẩn hóa,COUNT DISTINCT(product_id),sản phẩm,product_id không trống,Toàn bộ products hiện có,5.657000e+03,CALCULATED,products (1).csv,Đếm độc lập product_id duy nhất,
KPI14,Tổng bản ghi giá,Tổng snapshot trong phạm vi,COUNT(*),bản ghi,Không loại theo hợp lệ giá,2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,6.350000e+03,CALCULATED,price_snapshots (1).csv,Đếm số dòng snapshot,
KPI02,Giá thấp nhất,Giá hiện tại nhỏ nhất hợp lệ,MIN(current_price_vnd),VND,current_price_vnd hữu hạn > 0,2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,5.000000e+03,CALCULATED,price_snapshots (1).csv,Đối chiếu MIN trên tập hợp lệ,
KPI03,Giá cao nhất,Giá hiện tại lớn nhất hợp lệ,MAX(current_price_vnd),VND,current_price_vnd hữu hạn > 0,2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,3.160000e+06,CALCULATED,price_snapshots (1).csv,Đối chiếu MAX trên tập hợp lệ,
KPI04,Giá trung vị,Trung vị giá hiện tại hợp lệ,MEDIAN(current_price_vnd),VND,current_price_vnd hữu hạn > 0,2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,2.440000e+05,CALCULATED,price_snapshots (1).csv,Đối chiếu MEDIAN,
KPI08,Giá trung bình,Trung bình giá hiện tại hợp lệ,SUM(P_i)/n,VND,current_price_vnd hữu hạn > 0,2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,3.168619e+05,CALCULATED,price_snapshots (1).csv,Đối chiếu SUM/n,
KPI12,Cross-source Price Gap,Mean gap across verified equivalent product-date groups,Per verified group/date: MAX(P across retailers)-MIN(P across retailers); card=MEAN(group gaps),VND,"matched + human verified; match_group_id; >=2 retailers; same local date, latest valid snapshot per retailer",2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,NaN,NOT CALCULABLE,product_matching.csv + price_snapshots (1).csv,Independent min/max within each verified equivalent group,No verified equivalent product matches are currently available for cross-retailer comparison.
KPI05,Tỷ lệ bản ghi đủ dữ liệu giá,Tỷ lệ snapshot có giá hiện tại hợp lệ,N(current_price_vnd>0)/N(snapshot),%,Mẫu số là mọi snapshot trong phạm vi,2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,1.000000e+00,CALCULATED,price_snapshots (1).csv,Đếm tử và mẫu độc lập,
KPI06,Successful Matching Rate,Verified successful matches / formally defined records requiring matching,N_verified_successful_matches / N_need_matching,%,N_need_matching has not been formally defined; no candidate proxy denominator,Trạng thái file matching hiện tại,NaN,NOT CALCULABLE,product_matching.csv,Matching denominator (N_need_matching) has not been formally defined.,Matching denominator (N_need_matching) has not been formally defined.
KPI09,Tỷ lệ giảm giá tính lại trung bình,Trung bình tỷ lệ giảm tính lại từng snapshot,AVG((L_i-P_i)/L_i),%,Finite P>0 and L>0 and L>=P; anomalous L<P excluded,2026-10-18 đến 2026-10-21 (UTC+7); toàn bộ snapshot,2.777142e-01,CALCULATED,price_snapshots (1).csv,Tính lại 4861 cặp giá; anomalous_discount=0,


Cross-source Price Gap chỉ tính theo verified equivalent group và ngày với ít nhất 2 retailer. Overall observed price range chỉ ở EDA. Successful Matching Rate chưa có N_need_matching chính thức nên NOT CALCULABLE. Supporting Review Metrics – Hasaki & Guardian (tổng và từng nguồn) nằm trong sheet riêng.
Source product-match flag: this source field is NOT a verified link to price products. Unknown rate chỉ đếm dòng mang nhãn unknown trong file nguồn; không phải verified price matching rate. Supporting review metrics tách biệt với 10 core price KPI.


In [3]:
by_id=kpi.set_index('kpi_id')
assert len(kpi)==10
if g['gaps'].empty:
    assert pd.isna(by_id.loc['KPI12','current_value'])
    assert by_id.loc['KPI12','status']=='NOT CALCULABLE'
assert pd.isna(by_id.loc['KPI06','current_value'])
assert by_id.loc['KPI06','status']=='NOT CALCULABLE'
display(pd.DataFrame(g['review_support_rows']).rename(columns={'unknown':'source_flag_unknown_n','unknown_rate':'source_flag_unknown_rate'}))
print('Đối chiếu định nghĩa và trạng thái KPI: PASS')

Đối chiếu định nghĩa và trạng thái KPI: PASS


source,total,products,mean,median,purchased,images,source_flag_unknown_n,source_flag_unknown_rate
all,6673,749,4.899820,5.0,0.744942,0.161696,410,0.061442
guardian,410,193,4.928049,5.0,1.000000,0.109756,410,1.000000
hasaki,6263,556,4.897972,5.0,0.728245,0.165097,0,0.000000
